# einops-einsum — worked example 2: Weighted column sum (vector-times-matrix)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-einsum`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import einsum

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Contracting a 1-D weight vector against the matching axis of a 2-D matrix produces a weighted sum along that axis. The shared index appears in both operands but not on the output, so einsum sums over it — this is the row-vector × matrix product `w @ M`.

## Worked solution

We have weights `w` of shape `(i,)` and a matrix `M` of shape `(i, j)`, and want `out[j] = sum_i w[i] * M[i, j]`.

1. **Match the contracted axis.** Both `w` and `M` share the row index `i`. We write `'i, i j'` so einsum aligns `w[i]` with `M[i, j]` along `i`.
2. **Decide what survives.** The output should be indexed only by `j`, so we write `-> j`. Since `i` is absent from the output, it is summed; since `j` is present, it is preserved.
3. **Interpretation.** For each column `j`, we take the dot product of the weight vector with that column — a weighted column sum. This is exactly `w @ M` (a `(i,) @ (i, j) -> (j,)` product).

The takeaway: a length-`i` vector contracted against the `i` axis of a matrix is a weighted reduction over rows, leaving one value per column.

In [ ]:
def weighted_colsum(w: Tensor, M: Tensor) -> Tensor:
    return einsum(w, M, 'i, i j -> j')


t.manual_seed(0)
w = t.randn(4)
M = t.randn(4, 3)
out = weighted_colsum(w, M)
print('einsum    =', out)
print('reference =', w @ M)
print('close?    ', t.allclose(out, w @ M, atol=1e-5))